# Zstandard - Rust

All 8 Rust examples from [docs/zstd.md](https://platob.github.io/yggdryl/zstd/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::zstd;

let frame = zstd::dump(b"symbol,price\nAAPL,1\n")?;
assert_eq!(zstd::load(&frame)?, b"symbol,price\nAAPL,1\n");

In [ ]:
use yggdryl::zstd;

// Repetition is what zstd removes.
let payload = "AAPL,1\n".repeat(64);
let frame = zstd::dump(payload.as_bytes())?;
assert!(frame.len() < payload.len());

// Framing costs bytes, so a short payload comes out larger than it went in.
assert!(zstd::dump(b"AAPL,1\n")?.len() > 7);

// A payload that is not a frame is reported, not silently returned.
assert!(zstd::load(b"definitely not a compressed payload").is_err());

## Streams

In [ ]:
use std::io::{Read, Write};
use yggdryl::zstd;

let payload = "AAPL,1\n".repeat(64);

let mut encoded = Vec::new();
let mut encoder = zstd::writer(&mut encoded);
encoder.write_all(payload.as_bytes())?;
encoder.finish()?;

let mut decoded = Vec::new();
zstd::reader(encoded.as_slice()).read_to_end(&mut decoded)?;
assert_eq!(decoded, payload.as_bytes());

## Levels

In [ ]:
use yggdryl::{Level, zstd};

let payload = "AAPL,1\n".repeat(64);

for level in [Level::NONE, Level::FAST, Level::DEFAULT, Level::BEST] {
    let frame = zstd::dump_with_level(payload.as_bytes(), level)?;
    assert_eq!(zstd::load(&frame)?, payload.as_bytes(), "{level}");
}

// The scale is 0 to 9, and anything above it clamps.
assert_eq!(Level::DEFAULT.get(), 6);
assert_eq!(Level::new(12), Level::BEST);

In [ ]:
use std::io::Write;
use yggdryl::{Level, zstd};

let payload = "AAPL,1\n".repeat(64);

let mut encoded = Vec::new();
let mut encoder = zstd::writer_with_level(&mut encoded, Level::BEST);
encoder.write_all(payload.as_bytes())?;
encoder.finish()?;

assert_eq!(zstd::load(&encoded)?, payload.as_bytes());

## The transparent handle

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::zstd::{self, Zstd};

let mut handle = Zstd::new(Buffer::new());
handle.write_all_bytes(b"symbol,price\nAAPL,1\n")?;
handle.flush()?;

// The wrapper reads plain bytes and reports the decoded size.
assert_eq!(handle.read_all()?, b"symbol,price\nAAPL,1\n");
assert_eq!(handle.size(), 20);

// The wrapped handle holds the frame.
assert_eq!(
    zstd::load(handle.handle().as_slice())?,
    b"symbol,price\nAAPL,1\n"
);

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::zstd::Zstd;

// Wrapping touches nothing, and a handle with no bytes decodes to nothing.
let handle = Zstd::new(Buffer::new());
assert!(handle.read_all()?.is_empty());
assert_eq!(handle.size(), 0);

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::zstd::{self, Zstd};
use yggdryl::Level;

let payload = "AAPL,1\n".repeat(64);

let mut handle = Zstd::new(Buffer::new()).with_level(Level::BEST);
assert_eq!(handle.level(), Level::BEST);
handle.write_all_bytes(payload.as_bytes())?;

// Consuming the handle publishes the pending write first.
let inner = handle.into_handle()?;
assert_eq!(zstd::load(inner.as_slice())?, payload.as_bytes());